# Fine-tuned baselines and the protocol ladder

Everything else in this project runs on a CPU. This notebook exists for the one thing that
cannot: fine-tuning a CNN backbone.

## What it answers

Experiment E1 varied the train/test split while holding a **frozen linear head** fixed and
found no significant leakage inflation — macro-F1 0.5496 vs 0.5444, McNemar p = 0.75. That
null had an obvious hole: a linear head has almost no capacity to memorise, and leakage is
exploited *by* memorisation.

E1b tested that on the CPU by sweeping capacity instead of compute. Inflation rose with the
ability to memorise, and a 1-NN retriever — a pure memoriser — gained **+0.045 macro-F1** at
p ≈ 0 from ignoring patient identity, stable across three seeds.

**This notebook closes the loop with a realistic model.** A fine-tuned ResNet50 can memorise
and is what people actually deploy. If leakage inflation shows up here too, the finding is
no longer an artefact of any one model class.

Everything else — absolute accuracy, comparability with published numbers — is a bonus.

## Before you start

1. Run `python scripts/build_colab_bundle.py` locally. It writes `colab_bundle.zip`
   (~200 MB): images already resized to 384×384, plus the split manifests. The raw ODIR
   folder is ~7 GB; almost all of it is resolution the training run discards.
2. Upload that **one file** to `MyDrive/ophthalmic-screening-research/`.
3. Runtime → Change runtime type → **T4 GPU**.
4. Run all cells.

Expect roughly 1½–2 hours. Every fold checkpoints, so a disconnect costs one fold, not the
run.

## 1. Environment

In [ ]:
import os, sys, json, time, zipfile, platform, shutil
import torch

DRIVE_ROOT = '/content/drive/MyDrive/ophthalmic-screening-research'
WORK       = '/content/work'

from google.colab import drive
drive.mount('/content/drive')

print('GPU     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('PyTorch :', torch.__version__, '| CUDA', torch.version.cuda)
print('Python  :', platform.python_version())

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')

os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

## 2. Unpack the bundle

Extracted to local disk, not Drive — Drive I/O is slow enough to dominate training.

In [ ]:
bundle = f'{DRIVE_ROOT}/colab_bundle.zip'
assert os.path.exists(bundle), f'upload colab_bundle.zip to {DRIVE_ROOT} first'

if not os.path.exists(f'{WORK}/bundle.json'):
    os.makedirs(WORK, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(bundle) as z:
        z.extractall(WORK)
    print(f'extracted in {time.time()-t0:.0f}s')
else:
    print('already extracted')

meta = json.load(open(f'{WORK}/bundle.json'))
print(json.dumps(meta, indent=2))
print('images on disk:', len(os.listdir(f'{WORK}/images')))

## 3. Configuration

In [ ]:
SEED        = 42
IMAGE_SIZE  = 384
BATCH_SIZE  = 24
EPOCHS      = 8
PATIENCE    = 3
BACKBONE    = 'resnet50'      # matches the frozen-feature model E1/E1b used
LABELS      = ['N', 'D', 'G', 'C']

# The two rungs the headline comparison needs. Adding 'splits_patient_disjoint.csv'
# separates deduplication from patient grouping, at the cost of another ~45 minutes.
RUNGS = ['splits_image_level.csv', 'splits_patient_disjoint_dedup.csv']

RESULTS = f'{DRIVE_ROOT}/results'
CKPT    = f'{DRIVE_ROOT}/checkpoints'
print('rungs:', RUNGS)
print(f'{BACKBONE} @{IMAGE_SIZE}, batch {BATCH_SIZE}, up to {EPOCHS} epochs, patience {PATIENCE}')

## 4. Data

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

torch.manual_seed(SEED); np.random.seed(SEED)

TRAIN_TF = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
EVAL_TF = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class Fundus(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(f"{WORK}/images/{row['original_filename']}").convert('RGB')
        return self.transform(img), LABELS.index(row['label'])

def check(split_file):
    frame = pd.read_csv(f'{WORK}/splits/{split_file}')
    spanning = (frame.groupby('patient_id')['fold'].nunique() > 1).sum()
    kind = 'LEAKY (by design)' if 'image_level' in split_file else 'patient-disjoint'
    print(f"{split_file:<38}{len(frame):>6} images  {frame.patient_id.nunique():>5} patients"
          f"  {spanning:>5} patients span folds   {kind}")
    if 'image_level' not in split_file:
        assert spanning == 0, 'a patient spans folds in a split that should be disjoint'
    return frame

for r in RUNGS:
    check(r)

## 5. Model and training loop

Two-phase: head only, then the top blocks unfrozen at a lower learning rate — the same recipe the project's baseline used.

In [ ]:
import torch.nn as nn
from torchvision import models
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

def build():
    m = models.resnet50(weights='IMAGENET1K_V2')
    m.fc = nn.Linear(m.fc.in_features, len(LABELS))
    return m.cuda()

def epoch_pass(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    preds, trues, total = [], [], 0.0
    for x, y in loader:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast('cuda'):
            out = model(x); loss = criterion(out, y)
        if training:
            optimizer.zero_grad(); scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        total += loss.item() * len(y)
        preds.append(out.argmax(1).cpu()); trues.append(y.cpu())
    return total / len(loader.dataset), torch.cat(preds), torch.cat(trues)

def train_fold(train_df, val_df, tag):
    tl = DataLoader(Fundus(train_df, TRAIN_TF), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(Fundus(val_df, EVAL_TF), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True)

    model = build()
    weights = compute_class_weight('balanced', classes=np.arange(len(LABELS)),
                                   y=[LABELS.index(l) for l in train_df['label']])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).cuda())
    scaler = torch.amp.GradScaler()

    # phase 1 - head only, so an untrained head cannot wreck the pretrained weights
    for p in model.parameters(): p.requires_grad_(False)
    for p in model.fc.parameters(): p.requires_grad_(True)
    opt = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
    for e in range(2):
        epoch_pass(model, tl, criterion, opt, scaler)
        print(f'    [{tag}] warmup {e+1}/2 done')

    # phase 2 - unfreeze layer4 and the head
    for p in list(model.layer4.parameters()) + list(model.fc.parameters()):
        p.requires_grad_(True)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best, bad, best_state = -1.0, 0, None
    for e in range(EPOCHS):
        t0 = time.time()
        epoch_pass(model, tl, criterion, opt, scaler)
        _, vp, vt = epoch_pass(model, vl, criterion)
        macro = float(f1_score(vt, vp, average='macro'))
        print(f'    [{tag}] epoch {e+1}/{EPOCHS}  val macro-F1 {macro:.4f}  ({time.time()-t0:.0f}s)')
        sched.step()
        if macro > best:
            best, bad = macro, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'    [{tag}] early stop'); break

    if best_state: model.load_state_dict(best_state)
    return model, best

## 6. The protocol ladder, fine-tuned

Each rung is cross-validated and its out-of-fold predictions pooled. **Completed folds are
saved to Drive and skipped on re-run**, so a disconnect costs one fold.

In [ ]:
def out_of_fold(split_file):
    frame = pd.read_csv(f'{WORK}/splits/{split_file}')
    tag = split_file.replace('splits_', '').replace('.csv', '')
    blocks = []

    for fold in sorted(frame.fold.unique()):
        cache = f'{RESULTS}/E3_fold_{tag}_{fold}.csv'
        if os.path.exists(cache):
            print(f'  fold {fold}: cached'); blocks.append(pd.read_csv(cache)); continue

        tr, te = frame[frame.fold != fold], frame[frame.fold == fold]
        print(f'  fold {fold}: train {len(tr)}  test {len(te)}')
        model, best = train_fold(tr, te, f'{tag} f{fold}')

        model.eval(); probs = []
        loader = DataLoader(Fundus(te, EVAL_TF), batch_size=BATCH_SIZE, num_workers=2)
        with torch.no_grad(), torch.autocast('cuda'):
            for x, _ in loader:
                probs.append(torch.softmax(model(x.cuda()).float(), 1).cpu())
        probs = torch.cat(probs).numpy()

        block = te[['image_id', 'patient_id', 'label']].copy()
        block['y_true'] = block['label']
        block['y_pred'] = [LABELS[i] for i in probs.argmax(1)]
        for i, c in enumerate(LABELS):
            block['p_' + c] = probs[:, i]
        block.to_csv(cache, index=False)
        blocks.append(block)
        del model; torch.cuda.empty_cache()

    pooled = pd.concat(blocks, ignore_index=True)
    pooled.to_csv(f'{RESULTS}/E3_predictions_{tag}.csv', index=False)
    return tag, pooled

pooled = {}
for rung in RUNGS:
    print(f'\n=== {rung} ===')
    tag, frame = out_of_fold(rung)
    pooled[tag] = frame
    print(f'  pooled {len(frame)} predictions  macro-F1 '
          f'{f1_score(frame.y_true, frame.y_pred, average="macro"):.4f}')

## 7. The answer

Same statistics as the CPU experiments: pooled out-of-fold predictions, exact McNemar on the paired subset, bootstrap resampled at **patient** level.

In [ ]:
from scipy.stats import binomtest

def macro(frame):
    return float(f1_score(frame.y_true, frame.y_pred, average='macro'))

def patient_bootstrap(frame, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = frame.patient_id.to_numpy()
    unique = np.unique(patients)
    index = {p: np.where(patients == p)[0] for p in unique}
    draws = []
    for _ in range(n):
        picked = rng.choice(unique, size=len(unique), replace=True)
        rows = np.concatenate([index[p] for p in picked])
        draws.append(macro(frame.iloc[rows]))
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

leaky, clean = pooled.get('image_level'), pooled.get('patient_disjoint_dedup')
summary = {}

for name, frame in pooled.items():
    lo, hi = patient_bootstrap(frame)
    summary[name] = {'n': len(frame), 'macro_f1': round(macro(frame), 4),
                     'ci_low': round(lo, 4), 'ci_high': round(hi, 4)}
    print(f'{name:<26} macro-F1 {macro(frame):.4f}   95% CI [{lo:.4f}, {hi:.4f}]')

if leaky is not None and clean is not None:
    shared = sorted(set(leaky.image_id) & set(clean.image_id))
    a = leaky.set_index('image_id').loc[shared]
    b = clean.set_index('image_id').loc[shared]
    ca = (a.y_true == a.y_pred).to_numpy()
    cb = (b.y_true == b.y_pred).to_numpy()
    a_only, b_only = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    inflation = macro(leaky) - macro(clean)

    summary['comparison'] = {'inflation': round(inflation, 4), 'p_value': float(p),
                             'discordant': a_only + b_only, 'paired_n': len(shared)}
    print(f'\nINFLATION from ignoring patient identity: {inflation:+.4f}  '
          f'(McNemar p = {p:.6f}, {a_only+b_only} discordant pairs)')
    print('\nfor comparison, on frozen features:')
    print('  logistic head  +0.0052  p = 0.75      (E1  - cannot memorise)')
    print('  1-NN retriever +0.0453  p ~ 0         (E1b - pure memoriser)')
    print(f'  fine-tuned     {inflation:+.4f}  p = {p:.4g}   (this run)')

summary['config'] = {'backbone': BACKBONE, 'image_size': IMAGE_SIZE, 'epochs': EPOCHS,
                     'seed': SEED, 'rungs': RUNGS}
json.dump(summary, open(f'{RESULTS}/E3_finetuned_ladder.json', 'w'), indent=2)
print(f'\nwritten {RESULTS}/E3_finetuned_ladder.json')

## 8. Bring it home

From `MyDrive/ophthalmic-screening-research/results/` download into the local repo's
`results/`:

- `E3_predictions_*.csv`
- `E3_finetuned_ladder.json`

Then locally:

```bash
python scripts/make_figures.py
python scripts/summarise_results.py
```

**How to read whatever comes out.** If the fine-tuned inflation is large and significant, it
confirms E1b with a realistic model and E1's null was purely an artefact of model capacity —
the headline becomes stronger. If it is small, that is also a real result: it would mean
leakage on this dataset is exploitable only by literal retrieval and not by a fine-tuned
CNN, which is worth reporting and narrows the claim.

Both outcomes go in the paper. The kill rule written before the run applies here too.